In [1]:
import xarray as xr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import os

In [2]:
colvos_ds = xr.open_dataset('sg195_UWSSC_Colvos_Mission_timeseries.nc')
display(colvos_ds)
print(colvos_ds.depth.values)
print(colvos_ds.time.values)

#Align time with sg_data_point and apply offset (if needed)
adjusted_time = pd.to_datetime(colvos_ds['time'].values) + pd.DateOffset(years=19, months=7, days=16)
colvos_ds = colvos_ds.assign_coords(time=('sg_data_point', adjusted_time))
colvos_ds.time.values[0]


<xarray.Dataset> Size: 361kB
Dimensions:                                   (gps_info: 21,
                                               sg_data_point: 1975,
                                               trajectory: 7, dive: 7)
Coordinates:
    ctd_time                                  (sg_data_point) datetime64[ns] 16kB ...
    ctd_depth                                 (sg_data_point) float32 8kB ...
    latitude                                  (sg_data_point) float32 8kB ...
    longitude                                 (sg_data_point) float32 8kB ...
  * trajectory                                (trajectory) int32 28B 2 3 ... 7 8
Dimensions without coordinates: gps_info, sg_data_point, dive
Data variables: (12/66)
    gps_info_dive_number                      (gps_info) int32 84B ...
    sg_data_point_dive_number                 (sg_data_point) int32 8kB ...
    log_gps_time                              (gps_info) datetime64[ns] 168B ...
    time                                      (sg_data_point) datetime64[ns] 16kB ...
    pressure                                  (sg_data_point) float32 8kB ...
    depth                                     (sg_data_point) float32 8kB ...
    ...                                        ...
    end_longitude                             (dive) float32 28B ...
    depth_avg_curr_east                       (dive) float32 28B ...
    depth_avg_curr_north                      (dive) float32 28B ...
    depth_avg_curr_qc                         (dive) |S1 7B ...
    latlong_qc                                (dive) |S1 7B ...
    glider                                    |S12 12B ...
Attributes: (12/47)
    project:                         ColvosTripleGlider
    title:                           Physical, biological, and chemical data ...
    summary:                         SG195 ColvosTripleGlider
    source:                          Seaglider SG195
    references:                      http://data.nodc.noaa.gov/accession/0092291
    processing_level:                1.12
    ...                              ...
    date_modified:                   2024-06-01T00:06:32Z
    uuid:                            1adcb12e-1faa-11ef-a72f-f57473c7a252
    base_station_version:            3.0.2
    base_station_micro_version:      0
    quality_control_version:         1.12
    Conventions:                     CF-1.6

[0.45914176 0.4990671  0.6088617  ... 1.2276984  0.7086724  0.19962627]
['2004-10-15T16:05:52.303999872' '2004-10-15T16:05:56.961999872'
 '2004-10-15T16:06:02.404999936' ... '2004-10-15T23:46:52.346999936'
 '2004-10-15T23:46:57.302000000' '2004-10-15T23:47:02.254000000']


np.datetime64('2024-05-31T16:05:52.303999872')

In [3]:
colvos_ds['PAR_470nm'] = colvos_ds['eng_wlbb2fl_sig470nm']
colvos_ds['particle_concentration_700nm'] = colvos_ds['eng_wlbb2fl_sig700nm']
colvos_ds['chlorophyll_695nm'] = colvos_ds['eng_wlbb2fl_sig695nm']
colvos_ds['dissolved_oxygen'] = colvos_ds['aanderaa4330_dissolved_oxygen']
colvos_ds['instrument_dissolved_oxygen'] = colvos_ds['aanderaa4330_instrument_dissolved_oxygen']

# add metadata
colvos_ds['PAR_470nm'].attrs['pre_cleaning_name'] = 'eng_wlbb2fl_sig470nm'
colvos_ds['particle_concentration_700nm'].attrs['pre_cleaning_name'] = 'eng_wlbb2fl_sig700nm'
colvos_ds['chlorophyll_695nm'].attrs['pre_cleaning_name'] = 'eng_wlbb2fl_sig695nm'
colvos_ds['dissolved_oxygen'].attrs['pre_cleaning_name'] = 'aanderaa4330_dissolved_oxygen'
colvos_ds['instrument_dissolved_oxygen'].attrs['pre_cleaning_name'] = 'aanderaa4330_instrument_dissolved_oxygen'


#Select the relevant variables
new_colvos_ds = colvos_ds[['time', 'depth', 'latitude', 'longitude','temperature', 'salinity', 'dissolved_oxygen', 'instrument_dissolved_oxygen', 'PAR_470nm', 'particle_concentration_700nm', 'chlorophyll_695nm']]


#Convert to DataFrame and save
new_colvos_ds.to_dataframe().reset_index().to_csv('sg195_Colvos_Mission_timeseries_cleaned.csv', index=False)
new_colvos_ds.to_netcdf('sg195_Colvos_Mission_timeseries_cleaned.nc')
display(new_colvos_ds)


<xarray.Dataset> Size: 118kB
Dimensions:                       (sg_data_point: 1975)
Coordinates:
    time                          (sg_data_point) datetime64[ns] 16kB 2024-05...
    latitude                      (sg_data_point) float32 8kB 47.49 ... 47.54
    longitude                     (sg_data_point) float32 8kB -122.4 ... -122.4
    ctd_time                      (sg_data_point) datetime64[ns] 16kB 2004-10...
    ctd_depth                     (sg_data_point) float32 8kB -0.07496 ... 0....
Dimensions without coordinates: sg_data_point
Data variables:
    depth                         (sg_data_point) float32 8kB 0.4591 ... 0.1996
    temperature                   (sg_data_point) float32 8kB 11.5 ... 11.74
    salinity                      (sg_data_point) float32 8kB nan nan ... nan
    dissolved_oxygen              (sg_data_point) float32 8kB nan nan ... nan
    instrument_dissolved_oxygen   (sg_data_point) float32 8kB nan nan ... nan
    PAR_470nm                     (sg_data_point) float32 8kB 102.0 ... 81.0
    particle_concentration_700nm  (sg_data_point) float32 8kB 402.0 ... 259.0
    chlorophyll_695nm             (sg_data_point) float32 8kB 211.0 ... 95.0
Attributes: (12/47)
    project:                         ColvosTripleGlider
    title:                           Physical, biological, and chemical data ...
    summary:                         SG195 ColvosTripleGlider
    source:                          Seaglider SG195
    references:                      http://data.nodc.noaa.gov/accession/0092291
    processing_level:                1.12
    ...                              ...
    date_modified:                   2024-06-01T00:06:32Z
    uuid:                            1adcb12e-1faa-11ef-a72f-f57473c7a252
    base_station_version:            3.0.2
    base_station_micro_version:      0
    quality_control_version:         1.12
    Conventions:                     CF-1.6

In [4]:
colvos_ds = xr.open_dataset('sg195_UWSSC_Colvos_Mission_timeseries.nc')

#Apply time apply offset (if needed)
adjusted_start = pd.to_datetime(colvos_ds['start_time'].values) + pd.DateOffset(years=19, months=7, days=16)
adjusted_end = pd.to_datetime(colvos_ds['end_time'].values) + pd.DateOffset(years=19, months=7, days=16)
colvos_ds = colvos_ds.assign_coords(start_time=('dive', adjusted_start), end_time=('dive', adjusted_end))

colvos_ds['U_DAC'] = colvos_ds['depth_avg_curr_east']
colvos_ds['V_DAC'] = colvos_ds['depth_avg_curr_north']

# add metadata
colvos_ds['U_DAC'].attrs['pre_cleaning_name'] = 'depth_avg_curr_east'
colvos_ds['V_DAC'].attrs['pre_cleaning_name'] = 'depth_avg_curr_north'

#Select the relevant variables
new_colvos_ds = colvos_ds[['U_DAC', 'V_DAC', 'start_time', 'end_time', 'start_latitude', 'end_latitude', 'start_longitude', 'end_longitude']]
display(new_colvos_ds)

#Convert to DataFrame and save
new_colvos_ds.to_dataframe().reset_index().to_csv('sg195_Colvos_Mission_timeseries_DAC_cleaned.csv', index=False)
new_colvos_ds.to_netcdf('sg195_Colvos_Mission_timeseries_DAC_cleaned.nc')


<xarray.Dataset> Size: 280B
Dimensions:          (dive: 7)
Coordinates:
    start_time       (dive) datetime64[ns] 56B 2024-05-31T16:04:35 ... 2024-0...
    end_time         (dive) datetime64[ns] 56B 2024-05-31T16:31:38 ... 2024-0...
Dimensions without coordinates: dive
Data variables:
    U_DAC            (dive) float32 28B ...
    V_DAC            (dive) float32 28B ...
    start_latitude   (dive) float32 28B ...
    end_latitude     (dive) float32 28B ...
    start_longitude  (dive) float32 28B ...
    end_longitude    (dive) float32 28B ...
Attributes: (12/47)
    project:                         ColvosTripleGlider
    title:                           Physical, biological, and chemical data ...
    summary:                         SG195 ColvosTripleGlider
    source:                          Seaglider SG195
    references:                      http://data.nodc.noaa.gov/accession/0092291
    processing_level:                1.12
    ...                              ...
    date_modified:                   2024-06-01T00:06:32Z
    uuid:                            1adcb12e-1faa-11ef-a72f-f57473c7a252
    base_station_version:            3.0.2
    base_station_micro_version:      0
    quality_control_version:         1.12
    Conventions:                     CF-1.6